# Atomic electronic structure — Python port

**Status: work in progress; numerical results are not validated.**

This notebook uses the functions in `modules.py`; it does not redefine them. The historical notebook is preserved under `archive/`. See `../validation/PLAN.md` for known blockers.


In [ ]:
from pathlib import Path
import sys

# Support launching Jupyter from either the project root or python/.
module_dir = Path.cwd() if (Path.cwd() / "modules.py").exists() else Path.cwd() / "python"
if not (module_dir / "modules.py").exists():
    raise RuntimeError("Start Jupyter from the project root or its python directory.")
sys.path.insert(0, str(module_dir.resolve()))

import numpy as np
import matplotlib.pyplot as plt
import modules


## Historical example — disabled pending repair

The function below preserves the original example parameters. It is intentionally not called: neutral and ion inputs need to be separated and the numerical functions repaired first. Defining this function or importing the module is **not** a successful solver run.


In [ ]:
def run_unvalidated_example():
    # K
    
    # Start with V_ee = 0
    
    # n=4
    Gauss4 = np.array([
        [-np.sqrt(3/7 - 2/7 * np.sqrt(6/5)), (18 + np.sqrt(30)) / 36],
        [np.sqrt(3/7 - 2/7 * np.sqrt(6/5)), (18 + np.sqrt(30)) / 36],
        [-np.sqrt(3/7 + 2/7 * np.sqrt(6/5)), (18 - np.sqrt(30)) / 36],
        [np.sqrt(3/7 + 2/7 * np.sqrt(6/5)), (18 - np.sqrt(30)) / 36]
    ])
    
    Gauss_mat = Gauss4
    
    # Parameters
    a0 = 1  # m
    H = 1  # Hartree
    parameter1 = 0.5 * H * a0**2  # h_bar^2/(2*m)
    parameter2 = 2 * parameter1 / a0  # e^2/(4*pi*epsilon0)
    
    epsilon0 = 1 / (4 * np.pi)
    charge = 1
    
    kord = 4
    h = 0.1
    h1 = 0.5
    xmin, xmax = 0, 10
    
    # Linear knot sequence
    xarray_P1 = np.arange(xmin, 5 + h, h)
    xarray_P2 = np.arange(5 + h1, xmax + h1, h1)
    xarray_P = np.concatenate((xarray_P1, xarray_P2))
    
    xarray_V = xarray_P.copy()
    
    x_ghost_former = xmin * np.ones(kord - 1)
    x_ghost_behind = xmax * np.ones(kord - 1)
    tknot_P = np.concatenate((x_ghost_former, xarray_P, x_ghost_behind))
    tknot_V = tknot_P.copy()
    
    eta = 0.4
    
    l0 = 0
    l1 = 1
    l2 = 0
    
    Z = 19  # He
    
    # Occupation numbers
    N10 = 2  # 1s
    N20 = 2  # 2s
    N21 = 6  # 2p
    N30 = 2  # 3s
    N31 = 6  # 3p
    N32 = np.array([1, 0])
    N40 = 0
    
    iteration = 30
    
    # Call the energy calculation function
    E_tot, ryo_K = modules.cal_ryo_E(eta, iteration, charge, epsilon0, N10, N20, N21, N30, N31, N32, N40, xarray_P, xarray_V, tknot_P, tknot_V, kord, Gauss_mat, parameter1, parameter2, l0, l1, l2, Z)
    
    E_tot_ion, ryo_K_ion = modules.cal_ryo_E(eta, iteration, charge, epsilon0, N10, N20, N21, N30, N31, N32, N40, xarray_P, xarray_V, tknot_P, tknot_V, kord, Gauss_mat, parameter1, parameter2, l0, l1, l2, Z)
    
    # Plotting
    plt.figure(figsize=(10, 8))
    
    # Subplot 1
    plt.subplot(2, 2, 1)
    plt.plot(xarray_P, ryo_K[1, :] * (4 * np.pi * xarray_P**2), linestyle='--', linewidth=2, label='3p_63d_1')
    plt.plot(xarray_P, ryo_K_ion[1, :] * (4 * np.pi * xarray_P**2), linestyle=':', linewidth=2, label='3p_63d_1+')
    
    plt.xlabel('r [a0]')
    plt.ylabel('ρ(r) × 4πr²')
    plt.title('(b)')
    plt.legend()
    
    # Check if integral of ryo is N_occ * e
    integral = modules.cal_integral_ryo(ryo_K[0, 1:], xarray_P[1:])
    N = integral / charge
    
    integral_ion = modules.cal_integral_ryo(ryo_K_ion[0, 1:], xarray_P[1:])
    N_ion = integral_ion / charge
    
    plt.show()
